# Tema 4 · Inspección y detección de anomalías

Seis actividades sincronizadas con el PDF. Ejecuta por orden y guarda una copia. CPU y bibliotecas del curso; sin descargas.

Construiremos una referencia estadística con imágenes correctas, fijaremos un umbral en validación y evaluaremos una sola vez en prueba. La clase positiva es **defecto simulado**; una alarma del modelo significa desviación respecto a la referencia, no prueba de defecto físico.

Las imágenes son dibujos de superficies alineadas con marcas sintéticas. No son mediciones de fabricación real. El mapa de desviación es una explicación visual, no una máscara de defecto validada. Autoencoders y métodos profundos se introducen conceptualmente en el PDF, pero no se ejecutan aquí.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# La región de interés (ROI) se define por la geometría del dibujo.
# No es una segmentación aprendida ni se adapta a movimientos posteriores.
yy, xx = np.mgrid[:64, :64]
material = (xx >= 8) & (xx < 56) & (yy >= 8) & (yy < 56) & ((xx-32)**2 + (yy-32)**2 > 7**2)
ROI = (xx >= 13) & (xx < 51) & (yy >= 13) & (yy < 51) & ((xx-32)**2 + (yy-32)**2 > 10**2)

def generar(seed, tipo="correcta"):
    rng = np.random.default_rng(seed)
    luz = rng.uniform(0.88, 1.12)
    textura = 0.018 * np.sin(yy * 1.4 + rng.uniform(0, 2*np.pi))
    imagen = np.where(material, 0.62 * luz + textura, 0.16)
    imagen = imagen + rng.normal(0, rng.uniform(0.014, 0.030), imagen.shape)
    defecto = np.zeros_like(ROI)
    if tipo != "correcta":
        candidatos = np.argwhere(ROI)
        cy, cx = candidatos[rng.integers(len(candidatos))]
        if tipo == "raya":
            defecto = (np.abs(yy-cy) <= 1) & (np.abs(xx-cx) <= 7) & ROI
            imagen[defecto] -= rng.uniform(0.16, 0.27)
        elif tipo == "poro":
            defecto = ((xx-cx)**2 + (yy-cy)**2 <= 1.5**2) & ROI
            imagen[defecto] -= rng.uniform(0.035, 0.10)
        else:
            raise ValueError("Tipo no reconocido")
    return {"id": seed, "imagen": np.clip(imagen, 0, 1).astype(np.float32),
            "tipo": tipo, "defecto": defecto, "y": int(tipo != "correcta")}

def lote(inicio, n_correctas, n_defectos):
    return ([generar(inicio+i) for i in range(n_correctas)] +
            [generar(inicio+n_correctas+i, "raya" if i % 2 == 0 else "poro")
             for i in range(n_defectos)])

entrenamiento = lote(1000, 60, 0)
validacion = lote(2000, 40, 40)
prueba = lote(3000, 60, 60)
for a, b in [(entrenamiento, validacion), (entrenamiento, prueba), (validacion, prueba)]:
    assert not ({e["id"] for e in a} & {e["id"] for e in b})
assert all(e["y"] == 0 for e in entrenamiento)

## Actividad 1 · ¿Qué consideramos normal? (10 min)

Explora una pieza correcta, una raya y un poro. La ROI es la zona interior donde inspeccionamos; excluye borde y agujero. Los defectos se generan dentro de ella. El modelo no evaluará defectos fuera de esta zona.

In [ ]:
ejemplos = [entrenamiento[0], validacion[40], validacion[41]]
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, e in zip(axes, ejemplos):
    ax.imshow(e["imagen"], cmap="gray", vmin=0, vmax=1)
    ax.contour(ROI, levels=[0.5], colors=["cyan"], linewidths=0.6)
    ax.set_title(e["tipo"])
    ax.axis("off")
plt.tight_layout()
plt.show()
for nombre, datos in [("Entrenamiento", entrenamiento), ("Validación", validacion), ("Prueba", prueba)]:
    print(nombre, len(datos), "imágenes; correctas:", sum(e["y"] == 0 for e in datos))

**Explica:** ¿por qué necesitamos piezas correctas con variación admisible de iluminación y textura? ¿Podemos garantizar encontrar cualquier defecto por aprender solo lo normal?

**Tu respuesta:** …

## Actividad 2 · Aprender una referencia (12 min)

Para reducir cambios globales de brillo, a cada imagen le restamos la mediana de intensidad de su ROI. Aprendemos la media y la desviación por posición usando **solo entrenamiento correcto**. Esto presupone imágenes alineadas. El suelo de 0.025 evita dividir por desviaciones diminutas; se fija antes de evaluar. La puntuación no es una probabilidad de defecto.

In [ ]:
def preparar(imagen):
    return imagen.astype(np.float32) - np.median(imagen[ROI])
normalizadas = np.stack([preparar(e["imagen"]) for e in entrenamiento])
media = normalizadas.mean(axis=0)
desviacion = np.maximum(normalizadas.std(axis=0, ddof=1), 0.025)

def mapa_desviacion(imagen):
    z = np.abs(preparar(imagen) - media) / desviacion
    return np.where(ROI, z, 0)

def puntuar(imagen):
    valores = mapa_desviacion(imagen)[ROI]
    k = max(1, int(np.ceil(0.01 * len(valores))))
    # Media del 1 % de píxeles con mayor desviación dentro de la ROI.
    return float(np.partition(valores, len(valores)-k)[-k:].mean())

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, e in zip(axes, ejemplos):
    im = ax.imshow(mapa_desviacion(e["imagen"]), cmap="inferno", vmin=0, vmax=10)
    ax.set_title(f'{e["tipo"]} · score {puntuar(e["imagen"]):.2f}')
    ax.axis("off")
fig.colorbar(im, ax=axes.tolist(), label="Desviación respecto a referencia", shrink=0.75)
plt.show()

**Comprueba:** compara la media de todo el mapa con la media del 1 % más alto. Un defecto pequeño puede diluirse en una media global; incluso la puntuación local puede omitir poros débiles. El suelo estadístico y el 1 % son hiperparámetros, no constantes universales.

**Tu respuesta:** …

## Actividad 3 · Elegir un umbral en validación (15 min)

Alarma si `score >= umbral`. Comparamos candidatos únicamente en validación. Consideramos dos penalizaciones hipotéticas por pieza: FP=1/FN=5 (priorizar no omitir) y FP=5/FN=1 (priorizar pocas alarmas). No son costes reales ni una política universal; el reparto 50/50 de esta validación también influye en la elección.

In [ ]:
def contar(y, scores, umbral):
    pred = np.asarray(scores) >= umbral
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

def tasas(c):
    return {"FPR": c["FP"]/(c["FP"]+c["TN"]) if c["FP"]+c["TN"] else 0.0,
            "Recall": c["TP"]/(c["TP"]+c["FN"]) if c["TP"]+c["FN"] else 0.0,
            "Precisión": c["TP"]/(c["TP"]+c["FP"]) if c["TP"]+c["FP"] else 0.0}

y_val = np.array([e["y"] for e in validacion])
s_val = np.array([puntuar(e["imagen"]) for e in validacion])
candidatos = np.linspace(s_val.min()-0.01, s_val.max()+0.01, 101)
filas = []
for t in candidatos:
    c = contar(y_val, s_val, t)
    filas.append({"Umbral": t, **c, **tasas(c),
                  "Penalización omisiones": c["FP"] + 5*c["FN"],
                  "Penalización alarmas": 5*c["FP"] + c["FN"]})
tabla = pd.DataFrame(filas)
# En empate elegimos el primer candidato: el umbral menor.
elegidas = tabla.loc[[tabla["Penalización omisiones"].idxmin(),
                     tabla["Penalización alarmas"].idxmin()]].copy()
elegidas.index = ["Priorizar no omitir", "Priorizar pocas alarmas"]
print(elegidas.round(3).to_string())
UMBRAL = float(elegidas.loc["Priorizar no omitir", "Umbral"])
print("Decisión fijada para prueba: priorizar no omitir. Umbral =", round(UMBRAL, 3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for valor, nombre in [(0, "Correctas"), (1, "Defectos")]:
    axes[0].hist(s_val[y_val == valor], bins=15, alpha=0.6, label=nombre)
axes[0].axvline(UMBRAL, color="black", linestyle="--", label="Umbral elegido")
axes[0].set(xlabel="Puntuación de anomalía", ylabel="Número de imágenes")
axes[0].legend()
axes[1].plot(tabla["Umbral"], tabla["FPR"], label="Falsas alarmas / correctas")
axes[1].plot(tabla["Umbral"], 1-tabla["Recall"], label="Omisiones / defectos")
axes[1].set(xlabel="Umbral", ylabel="Proporción", ylim=(-0.02, 1.02))
axes[1].legend()
plt.tight_layout()
plt.show()

**Modifica:** considera una tercera penalización usando la tabla de validación. Justifica qué decisión tomarías y sus consecuencias. Conserva el umbral fijado para la evaluación de referencia. Si decides cambiarlo, hazlo aquí antes de abrir prueba y registra la decisión.

**Tu respuesta:** …

## Actividad 4 · Abrir prueba con decisiones fijadas (12 min)

Aplicamos el umbral elegido sin volver a ajustar media, desviación o puntuación. La matriz tiene filas reales y columnas predichas. La clase positiva es defecto simulado; predicción positiva significa alarma.

**Límite de este modelo:** la decisión que prioriza no omitir puede generar demasiadas falsas alarmas para ser útil. El ejercicio no recomienda desplegarla. Si ningún umbral cumple los objetivos, necesitamos mejorar captura, datos o modelo; el ajuste del umbral no sustituye esa mejora.

In [ ]:
y_test = np.array([e["y"] for e in prueba])
s_test = np.array([puntuar(e["imagen"]) for e in prueba])
pred_test = s_test >= UMBRAL
cuentas_test = contar(y_test, s_test, UMBRAL)
print("Prueba:", cuentas_test)
print("Tasas:", {k: round(v, 3) for k, v in tasas(cuentas_test).items()})
ConfusionMatrixDisplay.from_predictions(y_test, pred_test, labels=[0, 1],
                                        display_labels=["Correcta / no alarma", "Defecto / alarma"],
                                        cmap="Blues")
plt.xlabel("Decisión: no alarma / alarma")
plt.ylabel("Referencia: correcta / defecto")
plt.title("Prueba independiente · umbral fijado")
plt.tight_layout()
plt.show()
por_tipo = []
for tipo in ["raya", "poro"]:
    ids = np.array([e["tipo"] == tipo for e in prueba])
    por_tipo.append({"Tipo": tipo, "Número": int(ids.sum()),
                     "Detectados": int(pred_test[ids].sum()),
                     "Omitidos": int((~pred_test[ids]).sum())})
print(pd.DataFrame(por_tipo).to_string(index=False))

**Interpreta:** distingue FPR (FP/correctas) de 1−precisión (FP/alarmas). Si no aparece ningún error de un tipo, no significa que su tasa real sea cero: tenemos pocas imágenes y un generador limitado. No vuelvas a elegir parámetros consultando prueba.

**Tu respuesta:** …

## Actividad 5 · Examinar errores y mapas (10 min)

Selecciona una falsa alarma y una omisión si existen; si falta algún caso, muestra un ejemplo de esa categoría real y decláralo. El mapa muestra desviaciones: **no delimita automáticamente un defecto confirmado**. Los mapas comparten escala para poder compararlos.

In [ ]:
fp_ids = np.flatnonzero((y_test == 0) & pred_test)
fn_ids = np.flatnonzero((y_test == 1) & ~pred_test)
seleccion = [int(fp_ids[0]) if len(fp_ids) else int(np.flatnonzero(y_test == 0)[0]),
             int(fn_ids[0]) if len(fn_ids) else int(np.flatnonzero(y_test == 1)[0])]
fig, axes = plt.subplots(2, 2, figsize=(9, 7))
for col, idx in enumerate(seleccion):
    e = prueba[idx]
    categoria = ("FP" if pred_test[idx] else "TN") if e["y"] == 0 else ("TP" if pred_test[idx] else "FN")
    axes[0, col].imshow(e["imagen"], cmap="gray", vmin=0, vmax=1)
    if e["defecto"].any():
        axes[0, col].contour(e["defecto"], levels=[0.5], colors=["cyan"], linewidths=0.7)
    axes[0, col].set_title(f'{categoria} · {e["tipo"]} · {s_test[idx]:.2f}')
    axes[1, col].imshow(mapa_desviacion(e["imagen"]), cmap="inferno", vmin=0, vmax=10)
    axes[1, col].set_title("Mapa de desviación (0–10)")
for ax in axes.ravel(): ax.axis("off")
plt.tight_layout()
plt.savefig("tema4_errores.png", dpi=150, bbox_inches="tight")
plt.show()

**Explica:** ¿el defecto es pequeño, débil o está en una zona variable? ¿La alarma puede venir de una variación permitida? El contorno cian es la referencia sintética, no una predicción. Propón una mejora, pero evalúala en datos nuevos después de desarrollarla con entrenamiento/validación.

**Tu respuesta:** …

## Actividad 6 · Cambios de captura y defectos raros (13 min)

Primer diagnóstico: desplazamos piezas correctas 4 píxeles a la derecha sin actualizar la ROI ni alinear. Conservamos el modelo y el umbral. No son ejemplos independientes de las imágenes originales, sino transformaciones para estudiar un fallo.

Segundo diagnóstico: un ejemplo aritmético independiente muestra por qué el porcentaje de defectos afecta a la proporción de alarmas correctas. No extrapola nuestras tasas a una fábrica.

In [ ]:
correctas_test = [e for e in prueba if e["y"] == 0]
desplazadas = []
for e in correctas_test:
    movida = np.full_like(e["imagen"], 0.16)
    movida[:, 4:] = e["imagen"][:, :-4]
    desplazadas.append(movida)
s_movidas = np.array([puntuar(im) for im in desplazadas])
print("Alarmas en correctas originales:", int(np.sum(s_test[y_test == 0] >= UMBRAL)), "/", len(correctas_test))
print("Alarmas tras moverlas sin alinear:", int(np.sum(s_movidas >= UMBRAL)), "/", len(desplazadas))
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
axes[0].imshow(correctas_test[0]["imagen"], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Correcta original")
axes[1].imshow(desplazadas[0], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Correcta desplazada")
axes[2].imshow(mapa_desviacion(desplazadas[0]), cmap="inferno", vmin=0, vmax=10)
axes[2].set_title("Desviación sin realinear")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()
# Ejemplo hipotético separado del experimento anterior.
n_piezas, prevalencia, recall_hipotetico, fpr_hipotetico = 10000, 0.01, 0.90, 0.05
n_defectos = n_piezas * prevalencia
n_correctas = n_piezas - n_defectos
tp = n_defectos * recall_hipotetico
fp = n_correctas * fpr_hipotetico
print(f"Ejemplo hipotético: {tp:.0f} alarmas por defectos y {fp:.0f} falsas alarmas.")
print(f"Fracción de alarmas correctas: {tp/(tp+fp):.1%}")

**Discute:** ¿reentrenarías inmediatamente con piezas desplazadas o revisarías primero el sistema de captura y alineación? ¿Por qué 90 % de recall no significa que el 90 % de alarmas sean defectos?

**Tu respuesta:** …

## Entrega

Incluye tabla de los dos criterios de umbral, justificación de una decisión, matriz y tasas de prueba, dos casos comentados (o ausencia de esa categoría), y diagnóstico de desplazamiento. Descarga `tema4_errores.png` antes de cerrar Colab.

**Autoevaluación:** puedo distinguir normalidad y conformidad, explicar FP/FN y sus denominadores, fijar un umbral sin usar prueba, y describir límites de un mapa de anomalía y de una evaluación sintética.